# Ferienakademie 2026 — shared project playground

This notebook is the student-facing entry point. The scenario and global rules are fixed. The main things to change are `setup()` and `act()`.

The example strategy below is deliberately simple, but it solves the easy playground and exercises the main mechanisms: local vision, memory, pheromones, energy, day/night, and cooperative pushing.

In [ ]:
import math
import numpy as np

from fa2026 import Action, Cell, Config, Pheromone, execute, load_scenario

scenario = load_scenario("playground")
scenario.rules

## 1. Define the strategy

`setup(rules)` is called once before the simulation. It can inspect the fixed rules and prepare an immutable configuration.

`act(observation, memory, agent_type, config)` is called once per agent and turn. The agent has no absolute position, no agent ID, and no explicit clock.

In [ ]:
def setup(rules):
    # One moderately persistent recruitment pheromone.
    # parameters = (step length, pheromone amount, night vision size, stand-off)
    return Config(
        pheromones=(Pheromone(decay=0.10),),
        initial_memory=((0.0, 0.0, 0.0, 0.0),),
        parameters=((
            0.42,
            0.18,
            2 * rules.night_vision_radius + 1,
            0.5 * rules.cargo_size + 0.15,
        ),),
    )


def visible_vector(observation, flag):
    """Vector to the mean center of visible cells carrying a bit flag."""
    mask = (observation.vision & int(flag)) != 0
    ys, xs = np.nonzero(mask)
    if xs.size == 0:
        return None

    center = observation.vision.shape[0] // 2
    dx = xs - center + 0.5 - observation.cell_position[0]
    dy = ys - center + 0.5 - observation.cell_position[1]
    return np.array([float(np.mean(dx)), float(np.mean(dy))])


def unit(vector):
    length = float(np.linalg.norm(vector))
    return vector / length if length > 0.0 else np.zeros(2)


def act(observation, memory, agent_type, config):
    m = memory.copy()
    step, emission, night_size, stand_off = config.parameters[agent_type]

    cargo = visible_vector(observation, Cell.CARGO)
    target = visible_vector(observation, Cell.TARGET)

    # The smaller vision window reveals that it is night.
    # This simple strategy rests and recharges at night.
    if observation.vision.shape[0] == int(night_size):
        return Action(), m

    # If cargo and target are visible, get behind the cargo and push it
    # toward the target using only local geometry.
    if cargo is not None and target is not None:
        push_direction = unit(target - cargo)
        behind_cargo = cargo - stand_off * push_direction

        along = float(np.dot(cargo, push_direction))
        lateral = cargo - along * push_direction
        ready_to_push = (
            np.linalg.norm(behind_cargo) < 0.65
            or (along > 0.0 and np.linalg.norm(lateral) < 0.45)
        )

        move = step * (push_direction if ready_to_push else unit(behind_cargo))
        return Action(move=tuple(move), pheromones=(emission,)), m

    # If only the cargo is visible, approach it and leave a recruitment trail.
    if cargo is not None:
        return Action(
            move=tuple(step * unit(cargo)),
            pheromones=(emission,),
        ), m

    # Follow a neighboring pheromone maximum if one is available.
    pheromone = observation.pheromones[0]
    if float(np.max(pheromone)) > 1e-8:
        py, px = np.unravel_index(np.argmax(pheromone), pheromone.shape)
        direction = np.array([px - 1, py - 1], dtype=float)
        if np.linalg.norm(direction) > 0.0:
            return Action(move=tuple(step * unit(direction))), m

    # Otherwise explore deterministically. Memory stores heading, a countdown,
    # and an initialization flag.
    if m[2] < 0.5:
        phase = (
            observation.cell_position[0]
            + 0.61803398875 * observation.cell_position[1]
        ) % 1.0
        m[0] = 2.0 * math.pi * phase
        m[1] = 8.0
        m[2] = 1.0

    angle = float(m[0])
    countdown = float(m[1]) - 1.0
    if countdown <= 0.0:
        angle = (angle + 2.399963229728653) % (2.0 * math.pi)
        countdown = 8.0

    move = step * np.array([math.cos(angle), math.sin(angle)])

    # Immediate wall avoidance.
    center = observation.vision.shape[0] // 2
    sx = 1 if move[0] > 0 else (-1 if move[0] < 0 else 0)
    sy = 1 if move[1] > 0 else (-1 if move[1] < 0 else 0)
    if sx and int(observation.vision[center, center + sx]) & int(Cell.WALL):
        move[0] *= -1
    if sy and int(observation.vision[center + sy, center]) & int(Cell.WALL):
        move[1] *= -1

    m[0] = math.atan2(move[1], move[0])
    m[1] = countdown
    return Action(move=tuple(move)), m

## 2. Watch one run

Display settings do not change the game. `draw_every` controls how often a frame is drawn.

In [ ]:
result = execute(
    scenario,
    setup,
    act,
    visualize=True,
    delay=0.0,
    draw_every=1,
    max_turns=500,
)
result

## 3. Run without visualization

Use this mode for faster testing and optimization. The score is the number of turns needed to place the cargo completely inside the target.

In [ ]:
result = execute(
    scenario,
    setup,
    act,
    visualize=False,
    max_turns=500,
)
print(result)

## Your turn

The playground is intentionally easy. More difficult scenarios can change the fixed rules and geometry while keeping exactly the same `setup()` / `act()` interface. Try to make the strategy faster, more energy-efficient in its decisions, and more robust to different conditions.